# 🌸 MakeupLLM 训练
Qwen2.5-VL-7B + LoRA 微调，妆容风格识别

**运行前**：菜单栏 → 运行时 → 更改运行时类型 → **T4 GPU**

In [ ]:
# Cell 1: 安装依赖
!pip install -q llamafactory[metrics] qwen-vl-utils modelscope
!pip install -q flash-attn --no-build-isolation
print('✅ 依赖安装完成')

In [ ]:
# Cell 2: 下载模型
from modelscope import snapshot_download
model_path = snapshot_download('Qwen/Qwen2.5-VL-7B-Instruct', local_dir='models/Qwen2.5-VL-7B')
print(f'✅ 模型下载完成: {model_path}')

In [ ]:
# Cell 3: 上传训练数据
# 运行后会弹出文件选择框，上传 train.json, eval.json, dataset_info.json
from google.colab import files
import os, shutil

os.makedirs('data', exist_ok=True)
os.makedirs('data/makeup_images', exist_ok=True)

print('请上传 train.json, eval.json, dataset_info.json:')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'  ✅ {fname}')

print('\n请上传 makeup_images.zip (图片压缩包):')
uploaded2 = files.upload()
for fname in uploaded2:
    shutil.move(fname, f'data/{fname}')
    if fname.endswith('.zip'):
        !cd data && unzip -q {fname} && rm {fname}
        print(f'  ✅ 解压完成')

# 验证
import json
with open('data/train.json') as f:
    train = json.load(f)
print(f'\n训练集: {len(train)} 条')
print(f'图片数: {len(os.listdir("data/makeup_images"))} 张')

In [ ]:
# Cell 4: 克隆 LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e ".[metrics]" -q

# 复制数据
!cp data/train.json LLaMA-Factory/data/
!cp data/eval.json LLaMA-Factory/data/
!cp data/dataset_info.json LLaMA-Factory/data/
!cp -r data/makeup_images LLaMA-Factory/data/
print('✅ LLaMA-Factory 配置完成')

In [ ]:
# Cell 5: 创建训练配置
import yaml

config = {
    'model_name_or_path': 'models/Qwen2.5-VL-7B',
    'template': 'qwen2_vl',
    'trust_remote_code': True,
    'stage': 'sft',
    'do_train': True,
    'finetuning_type': 'lora',
    'lora_target': 'all',
    'lora_rank': 16,
    'lora_alpha': 16,
    'dataset': 'makeup_vl_train',
    'eval_dataset': 'makeup_vl_eval',
    'cutoff_len': 2048,
    'overwrite_cache': True,
    'preprocessing_num_workers': 4,
    'output_dir': 'saves/makeup-llm/lora',
    'logging_steps': 10,
    'save_steps': 100,
    'save_total_limit': 3,
    'plot_loss': True,
    'overwrite_output_dir': True,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1.0e-4,
    'num_train_epochs': 3.0,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.1,
    'bf16': True,
    'flash_attn': 'fa2',
    'gradient_checkpointing': True,
}

with open('LLaMA-Factory/train_makeup.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)
print('✅ 训练配置已创建')
print(yaml.dump(config, allow_unicode=True))

In [ ]:
# Cell 6: 开始训练！
import datetime
print(f'🚀 训练开始: {datetime.datetime.now()}')
print('=' * 50)

!cd LLaMA-Factory && llamafactory-cli train train_makeup.yaml

print('=' * 50)
print(f'✅ 训练完成: {datetime.datetime.now()}')

In [ ]:
# Cell 7: 导出模型（可选）
merge_config = {
    'model_name_or_path': 'models/Qwen2.5-VL-7B',
    'adapter_name_or_path': 'saves/makeup-llm/lora',
    'template': 'qwen2_vl',
    'finetuning_type': 'lora',
    'export_dir': 'models/MakeupLLM-7B',
    'export_size': 5,
    'export_device': 'cpu',
}
with open('LLaMA-Factory/merge.yaml', 'w') as f:
    yaml.dump(merge_config, f, default_flow_style=False)

!cd LLaMA-Factory && llamafactory-cli export merge.yaml
print('✅ 模型导出完成')

In [ ]:
# Cell 8: 打包下载模型
!cd models && tar czf /content/MakeupLLM-7B.tar.gz MakeupLLM-7B/
from google.colab import files
files.download('/content/MakeupLLM-7B.tar.gz')
print('✅ 模型已打包，开始下载')